# Session 2: Descriptive statistics, hypothesis tests, linear regression

**TOPICS:** Summarizing data, Measures of central tendency, pandas data manipulation, *[Break]* Covariation, Correlations and t-tests, Simple linear regressions.

### Summarise my data

In [ ]:
from pathlib import Path
import calendar

import pandas as pd

MONTHS = list(calendar.month_name)[1:]   # ["January", "February", ..., "December"]


def find_data_file(name):
    """Look for data/<name> in this folder, in the folders above it, and one level below."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        candidate = folder / "data" / name
        if candidate.exists():
            return candidate
    for candidate in sorted(here.glob(f"*/data/{name}")):
        return candidate
    raise FileNotFoundError(
        f"Could not find data/{name}. Python is currently looking from: {here}\n"
        "Fix: put the 'data' folder next to this notebook (or in a folder above it)."
    )

def load_seatbelts():
    """The monthly data frame we build in Session 1 -- rebuilt on the spot if the file is missing."""
    try:
        df = pd.read_csv(find_data_file("seatbelts_monthly.csv"), parse_dates=["date"])
    except FileNotFoundError:
        try:
            df = pd.read_csv(find_data_file("Seatbelts.csv"))
        except FileNotFoundError:
            import statsmodels.api as sm   # last resort: download the same data (needs internet)
            try:
                df = sm.datasets.get_rdataset("Seatbelts", "datasets").data
            except Exception as err:
                raise FileNotFoundError(
                    "Could not find data/Seatbelts.csv and could not download it either. "
                    "Keep the 'data' folder next to the notebooks (or connect to the internet)."
                ) from err
        df["date"] = pd.date_range("1969-01-01", periods=len(df), freq="MS")
        df["month"] = df["date"].dt.month_name()
    df["month"] = pd.Categorical(df["month"], categories=MONTHS, ordered=True)
    return df


# let's load the data-set we built last time. If the saved file can't be found (wrong working
# directory? Session 1 not finished?), load_seatbelts() rebuilds the same data frame from the raw data instead.
import numpy as np
from scipy import stats

seatbelts_monthly = load_seatbelts()
seatbelts_monthly.head()

In [ ]:
# let's look at it to remember its structure
seatbelts_monthly.describe()

In [ ]:
seatbelts_monthly["law"].describe()    # we can get summary info of individual variables, too

In [ ]:
# In R there was a trick, attach(), to avoid typing the data frame's name all the time.
# Python has no attach() -- and that's a feature: it is always clear which data frame a column comes from.
# The nearest thing is pulling columns out into short names:
deaths = seatbelts_monthly["DriversKilled"]
month = seatbelts_monthly["month"]
law = seatbelts_monthly["law"]

print(month.unique())
print(list(seatbelts_monthly.columns))

In [ ]:
# How many observations in each month?
print(month.value_counts().sort_index())

In [ ]:
# What is the count of law==1 (post-law) months in each calendar month?
pd.crosstab(law, month)      # this is called cross-tabulation

In [ ]:
# It is a bit more verbose when one of our variables is continuous: one row per distinct value of DriversKilled
crosstab_deaths = pd.crosstab(deaths, month)
print(crosstab_deaths.shape)
crosstab_deaths.head()

In [ ]:
# It might be more useful to see if the mean of DriversKilled changes by month
seatbelts_monthly.groupby("month", observed=True)["DriversKilled"].mean().round(1)
# NB: this is a real, useful finding, not just a syntax demo -- deaths are far higher in
# Oct/Nov/Dec than in spring. Any comparison that mixes different seasonal compositions
# between two groups risks confusing seasonality with whatever we're actually comparing.
# We'll come back to exactly this point in the hypothesis-testing section below.

### Measures of central tendency *(syntax demo)*

In [ ]:
# for teaching purposes, let's pretend a handful of months' petrol price wasn't reported.
# (Seatbelts itself has no missing data, so we make a COPY of the column and punch a few gaps
# in it -- the real PetrolPrice column stays intact for the models later on.)
rng = np.random.default_rng(2)
seatbelts_monthly["PetrolPrice_reported"] = seatbelts_monthly["PetrolPrice"]
gaps = rng.choice(len(seatbelts_monthly), size=6, replace=False)
seatbelts_monthly.loc[seatbelts_monthly.index[gaps], "PetrolPrice_reported"] = np.nan

# Mean: the average of a set of numbers
mean_deaths = deaths.mean()
print(mean_deaths)
# Print the mean more aesthetically
print(f"Mean: {mean_deaths:.2f}")

# Median: the middle value in a set of numbers.
median_deaths = deaths.median()
print(f"Median: {median_deaths}")

In [ ]:
# Mode: the most frequently occurring value in a set of numbers.
# pandas has .mode(), but it returns *all* tied values, sorted. A custom function that returns just
# the first most-frequent value (in order of appearance) is easy to write:
def calculate_mode(x):
    counts = pd.Series(x).dropna().value_counts(sort=False)   # counts, in order of first appearance
    return counts.idxmax()                                      # the first value with the highest count

mode_deaths = calculate_mode(deaths)
print(f"Mode: {mode_deaths}")
print(deaths.mode().tolist())          # pandas' version (a list, in case of ties)

In [ ]:
# NB: missing values are a problem for measures of central tendency, because they change the divisor
# (the number of observations). What happens to the mean of "PetrolPrice_reported"?
reported = seatbelts_monthly["PetrolPrice_reported"]
print(reported.isna().sum(), "missing values")
print("pandas default:      ", reported.mean())                    # pandas skips missing values silently...
print("skipna=False:        ", reported.mean(skipna=False))        # ...unless you tell it not to -> nan
print("NumPy on the values: ", np.mean(reported.to_numpy()))       # NumPy does NOT skip them -> nan
print("true mean (no gaps): ", seatbelts_monthly["PetrolPrice"].mean())

> **R → Python:** in R, `mean(x)` returns `NA` when `x` has missing values, and you must opt in to skipping them with `na.rm = TRUE`. pandas does the opposite: it skips missing values **by default** (`skipna=True`), which is convenient but means a gap in your data can disappear without a warning. NumPy sits with R: it propagates `nan`. Know which tool you are calling.

### Task: measures of central tendency on a new variable

`seatbelts_monthly` also has `VanKilled` — the number of van drivers killed each month.

1. Compute the mean, median, and mode of `VanKilled`.
2. Compare the three. Are they close together, or does one stand out?
3. In one sentence: does `VanKilled` look like a variable a mean summarizes well, or is it the kind of variable (a small count, possibly skewed) where you'd want to look at more than just the mean?

In [ ]:
# your code here

### More pandas *(syntax demo)*

pandas has a method for each everyday data-manipulation job, and methods can be chained one after another:

- `.query()` — filter rows by a condition
- `.sort_values()` — sort by one or more columns
- column selection and `.rename()` — choose and rename columns
- `.assign()` — create or transform columns
- `.groupby().agg()` — aggregate with functions like `mean`, `std` and `count`

In [ ]:
# Advanced filtering: multiple conditions
filtered_seatbelts = seatbelts_monthly.query("kms > 10000 and law == 0")
# Note: in .query() you write "and" / "or"; with square brackets you use & and | (and wrap each condition in parentheses)
filtered_seatbelts_2 = seatbelts_monthly[(seatbelts_monthly["kms"] > 10000) & (seatbelts_monthly["law"] == 0)]
print(filtered_seatbelts.shape, filtered_seatbelts_2.shape)

In [ ]:
# Sort by DriversKilled in descending order, then by date in ascending order
arranged_seatbelts = seatbelts_monthly.sort_values(["DriversKilled", "date"], ascending=[False, True])
arranged_seatbelts[["date", "DriversKilled"]].head()

In [ ]:
# Select date and DriversKilled columns, and rename DriversKilled to Deaths
selected_seatbelts = seatbelts_monthly[["date", "DriversKilled"]].rename(columns={"DriversKilled": "Deaths"})
selected_seatbelts.head()

In [ ]:
# Add a new column 'AboveMeanDeaths' based on DriversKilled, and bucket DriversKilled into quartiles
mutated_seatbelts = seatbelts_monthly.assign(
    AboveMeanDeaths=lambda d: np.where(d["DriversKilled"] >= d["DriversKilled"].mean(), "Yes", "No"),
    DeathsQuartile=lambda d: pd.qcut(d["DriversKilled"], 4, labels=["Low", "Medium", "High", "Very High"]),
)

print(mutated_seatbelts["DeathsQuartile"].value_counts().sort_index())
print(mutated_seatbelts["AboveMeanDeaths"].value_counts())

In [ ]:
# Aggregating: the mean, standard deviation and count of deaths, before and after the law
seatbelts_monthly.groupby("law")["DriversKilled"].agg(["mean", "std", "count"]).round(1)

### Manipulate my data *(syntax demo)*

In [ ]:
# create new variables
new_seatbelts = seatbelts_monthly.assign(kms_x2=seatbelts_monthly["kms"] * 2)
new_seatbelts = seatbelts_monthly.assign(lag_deaths=seatbelts_monthly["DriversKilled"].shift(1))   # you can name the variable you create
# add (or replace) a column by assigning to its name
new_seatbelts["lag2_deaths"] = new_seatbelts["DriversKilled"].shift(2)
new_seatbelts[["date", "DriversKilled", "lag_deaths", "lag2_deaths"]].head()   # shift() leaves gaps at the start

In [ ]:
# creating new empty variables
# this comes in handy when you want to initiate loops
new_seatbelts["empty_text"] = ""          # a new empty character column
new_seatbelts["empty_number"] = np.nan    # a new empty numerical column

# create a variable based on multiple conditions
# in this case, on the value of another column
winter_months = ["December", "January", "February"]
new_seatbelts["season"] = np.select(
    [new_seatbelts["month"].isin(winter_months)],   # list of conditions (checked in order: put the most specific first)
    ["Winter"],                                     # the value for each condition
    default="Not winter",                           # everything else
)
# for more complex text selection look into regular expressions (the re module, or .str.contains)

# searching and replacing part of a string with another string
new_seatbelts["x"] = new_seatbelts["season"].str.replace("Not winter", "Rest of year", regex=False)
# there is a whole universe out there on string handling (see also: regex, text encoding, etc) but we're not going there now
new_seatbelts[["month", "season", "x"]].head(3)

In [ ]:
# converting text columns to something more useful, e.g. categoricals are for data that take a set number of
# predefined values. You can control their order.
print(seatbelts_monthly["month"].dtype)
new_seatbelts["season"] = new_seatbelts["season"].astype("category")
print(new_seatbelts["season"].dtype, list(new_seatbelts["season"].cat.categories))

# selecting/dropping columns
mini_seatbelts = new_seatbelts[["date", "law"]]
new_seatbelts = new_seatbelts.drop(columns="kms")
print(mini_seatbelts.shape, new_seatbelts.shape)

### Variance, covariance, and correlation

As you know from your stats background, computing the variance, covariance and correlation between your variables of interest by hand is tedious (and you'd need to define other quantities of interest in the process, too, e.g. the standard deviation). Python can make our life easy!

In [ ]:
print(list(seatbelts_monthly.columns))
# these only work on numeric values

# Calculate variance
# Intuition: variance measures the dispersion of a single variable. It quantifies how much the values of a variable differ from the mean of the variable.
variance = seatbelts_monthly["DriversKilled"].var()
print(f"Variance: {variance:.2f}")
# "High" compared to what? Variance alone doesn't say -- it's in SQUARED units (here,
# deaths^2), which nobody has good intuition for. Two fixes:
# (1) take the square root to get the standard deviation, back in the original units:
sd_deaths = np.sqrt(variance)      # equivalently seatbelts_monthly["DriversKilled"].std()
print(f"SD: {sd_deaths:.2f}")
# (2) compare that SD to the mean, to get a sense of *relative* spread:
mean_deaths = seatbelts_monthly["DriversKilled"].mean()
print(f"SD as a % of the mean: {100 * sd_deaths / mean_deaths:.1f}")
# SD ~= 25 deaths, against a mean of ~123/month -- so a typical month's death count is
# about 25 deaths (~21%) above or below the average. THAT's what "high" means here: not
# a number in isolation, but a spread relative to the thing being measured.

> **R → Python:** the pandas method `.var()` divides by *n − 1* (the sample variance), like R's `var()`. But NumPy's `np.var()` divides by *n* by default. Same data, different answer, and no warning:

In [ ]:
print("pandas .var():            ", seatbelts_monthly["DriversKilled"].var())
print("NumPy np.var() (default): ", np.var(seatbelts_monthly["DriversKilled"].to_numpy()))
print("NumPy with ddof=1:        ", np.var(seatbelts_monthly["DriversKilled"].to_numpy(), ddof=1))

In [ ]:
# Calculate covariance
# Intuition: covariance measures the joint variability of two variables. It indicates the direction of the linear relationship between the variables.
covariance = seatbelts_monthly["kms"].cov(seatbelts_monthly["DriversKilled"])
print(f"Covariance: {covariance:.1f}")
# A positive covariance indicates that the two variables tend to move in the same direction (when one increases, the other tends to increase).
# A negative covariance indicates that the two variables tend to move in opposite directions (when one increases, the other tends to decrease).
# Unlike variance, covariance's MAGNITUDE isn't meaningfully "high" or "low" at all -- it
# depends on the arbitrary units of BOTH variables. Proof: if kms had been recorded in
# miles instead of kilometres, the same underlying relationship would give
# a completely different covariance number, just from rescaling:
kms_miles = seatbelts_monthly["kms"] * 0.621371
print(f"Same relationship, different units -> different covariance: {kms_miles.cov(seatbelts_monthly['DriversKilled']):.1f}")
# Only the SIGN of covariance is safe to read directly (negative here). For a magnitude
# you can actually compare across variables/units, you need correlation -- which is
# exactly why we compute it next.

In [ ]:
# Calculate correlation
# Intuition: it quantifies the degree to which two variables are related. It measures both the strength and direction of the linear relationship between two variables.
# The correlation coefficient, typically denoted as r, ranges from -1 to 1.
correlation = seatbelts_monthly["kms"].corr(seatbelts_monthly["DriversKilled"])
print(f"Correlation: {correlation:.3f}")
print(f"...and in miles: {kms_miles.corr(seatbelts_monthly['DriversKilled']):.3f}   <- unchanged by the choice of units")
# 1: Perfect positive linear relationship.
# -1: Perfect negative linear relationship.
# 0: No linear relationship.
# Notice the sign: it's NEGATIVE. Naively you might expect more driving (kms) to mean
# more deaths (more exposure to risk) -- but at this monthly, aggregate level, "kms" is
# also standing in for season (people drive more in fine-weather months, which are also
# the safer months). We'll see this exact confound resurface, and get resolved, in the
# regression task at the end of this session.

### Now we are moving to the realm of hypothesis testing...

Same as above, Python helps us with the otherwise computationally intense hypothesis testing (the `scipy.stats` module has the standard tests).

In [ ]:
# Comparing all 169 pre-law months against the 23 post-law months would mostly compare
# two very different eras (see Session 1) -- so we restrict to a narrower, more
# comparable window before testing anything.
seatbelts_window = seatbelts_monthly[seatbelts_monthly["date"] >= "1980-01-01"].copy()
print(seatbelts_window["law"].value_counts().sort_index())

# A small helper to print a t-test result readably
def show_ttest(result, label):
    low, high = result.confidence_interval()
    print(f"{label}: t = {result.statistic:.3f}, df = {result.df:.1f}, p = {result.pvalue:.4f}; "
          f"95% CI for the difference in means = [{low:.1f}, {high:.1f}]")

deaths_before = seatbelts_window.loc[seatbelts_window["law"] == 0, "DriversKilled"]
deaths_after = seatbelts_window.loc[seatbelts_window["law"] == 1, "DriversKilled"]
print(f"mean before = {deaths_before.mean():.1f}, mean after = {deaths_after.mean():.1f}")

In [ ]:
# One-tailed t-test
# Intuition: used when you have a specific direction of interest (e.g., testing if the mean of one group is greater than the mean of another group).
# equal_var=False gives Welch's t-test (does not assume equal variances) -- the same as R's default t.test().
one_tailed_ttest = stats.ttest_ind(deaths_before, deaths_after, equal_var=False, alternative="greater")
show_ttest(one_tailed_ttest, "One-tailed (before > after)")

# Two-tailed t-test
# Intuition: used when you are interested in any difference, regardless of direction (e.g., testing if the means of two groups are different).
two_tailed_ttest = stats.ttest_ind(deaths_before, deaths_after, equal_var=False, alternative="two-sided")
show_ttest(two_tailed_ttest, "Two-tailed")

# Paired t-test
# Intuition: Used when you have two related samples (e.g., measurements taken before and after a treatment on the same subjects).
# our data do not have repeated observations on the same unit (each month is a different
# unit), so I will just show you the function:
# stats.ttest_rel(sample_1, sample_2)

### A design detour: does seasonality change the picture?

The two-tailed test above compares 37 pre-law months against 23 post-law months, one month at a time. But we already saw (in "Summarise my data") that `DriversKilled` swings a lot by calendar month regardless of the law — so a month-by-month comparison is, in effect, also partly a comparison of *which months* happen to land on each side of the law change, not just of the law itself. One fix: aggregate to **yearly** means first, since averaging over 12 months cancels seasonality out.

In [ ]:
seatbelts_window["year"] = seatbelts_window["date"].dt.year

# groupby() is pandas' group-by-and-summarise: DriversKilled by year, averaged
yearly_deaths = seatbelts_window.groupby("year", as_index=False)["DriversKilled"].mean()
yearly_deaths["law"] = (yearly_deaths["year"] >= 1983).astype(int)   # the law came into force on 31 Jan 1983
print(yearly_deaths)

yearly_test = stats.ttest_ind(
    yearly_deaths.loc[yearly_deaths["law"] == 0, "DriversKilled"],
    yearly_deaths.loc[yearly_deaths["law"] == 1, "DriversKilled"],
    equal_var=False,
)
show_ttest(yearly_test, "Yearly means")
# Only 3 pre-law years vs 2 post-law years -- a t-test on 5 numbers has very little power,
# and shouldn't be over-interpreted as a rigorous test on its own. Treat this as a sanity
# check on DIRECTION and rough MAGNITUDE, not as a replacement for the monthly test above.
# Reassuringly, both approaches agree: deaths fall by a similar amount (~15/month)
# whether we look month-by-month or year-by-year. If the two approaches had disagreed
# sharply, that would be a red flag that seasonality (or something correlated with it)
# was doing more work than the law itself in the monthly comparison.

### Task: run your own t-test

`seatbelts_monthly` has `PetrolPrice` (the price of petrol; the dataset's documentation doesn't give units, so read it as a relative measure, not as pounds per litre).

1. Create a binary grouping variable: `"High"` if `PetrolPrice` is above its median, `"Low"` otherwise.
2. Run a two-tailed t-test comparing `DriversKilled` between the High and Low groups.
3. State your hypothesis in words *before* you look at the p-value, then interpret the result: is the difference statistically significant, and which group has fewer deaths?
4. One thing to flag (you don't need to resolve it): petrol prices generally rose over 1969-1984, the same period deaths were generally falling for other reasons too. Does that change how much you trust this comparison?

In [ ]:
# your code here

### Simple linear regression

We use `statsmodels`, whose **formula interface** reads almost exactly like R: `outcome ~ predictor1 + predictor2`. A column that is categorical (like `month`) is automatically turned into a set of 0/1 "dummy" variables, one per month except the first (January), which becomes the baseline.

In [ ]:
import statsmodels.formula.api as smf

mod1 = smf.ols("DriversKilled ~ kms", data=seatbelts_window).fit()
print(mod1.summary())

In [ ]:
print(list(seatbelts_window.columns))
# what shall we control for?
# we saw kms and DriversKilled were negatively correlated above, seemingly backwards --
# is that because kms is confounded with law, or with season (month), or both?
mod2 = smf.ols("DriversKilled ~ kms + law + PetrolPrice", data=seatbelts_window).fit()
print(mod2.summary())

In [ ]:
mod3 = smf.ols("DriversKilled ~ kms + law + PetrolPrice + month", data=seatbelts_window).fit()
print(mod3.summary())
# let's spend some time interpreting these models

### Side-by-side regression tables

Reading three `summary()` printouts one after another is slow. `summary_col` (from `statsmodels`) puts several models in one table, the way they appear in papers. (A separate package called `stargazer` does the same job and has a very similar API if you prefer its look.)

In [ ]:
from statsmodels.iolib.summary2 import summary_col

table = summary_col(
    [mod1, mod2, mod3],
    stars=True,
    float_format="%.3f",
    model_names=["(1)", "(2)", "(3)"],
    regressor_order=["kms", "law", "PetrolPrice", "Intercept"],   # only these rows are shown...
    drop_omitted=True,                                             # ...so the 11 month dummies are hidden
    info_dict={
        "Month dummies": lambda m: "Yes" if any("month" in name for name in m.params.index) else "No",   # ...but we say they are there
        "Observations": lambda m: f"{int(m.nobs)}",
    },
)
print(table)
# How to read it: each column is one model. The numbers are coefficients, the number in
# parentheses is the standard error, and the stars mark conventional significance levels
# (see the note under the table). Read left to right and watch what happens to `kms` and
# `law` as controls are added.
# The same object can be exported: table.as_html(), table.as_latex()

### Task: from bivariate to multivariate, one step at a time

Rather than jumping straight to `mod2`'s predictor list, build the bridge yourself:

1. Fit `mod_a` = `DriversKilled ~ kms + law` on `seatbelts_window` — one covariate added to `mod1`. How does the coefficient on `kms` change compared to `mod1`? Does its *sign* change?
2. Fit `mod_b` = `DriversKilled ~ kms + law + month` — controlling for calendar month directly, instead of aggregating to yearly like we did earlier. Has `kms`'s coefficient kept moving?
3. Compare R² across `mod1`, `mod_a`, `mod_b` (`model.rsquared`). What's happening to model fit as you add predictors — and which addition matters most?
4. In a sentence: what does it mean that `kms`'s coefficient flips sign once `month` is in the model? What was the raw correlation actually picking up?
5. Put `mod1`, `mod_a` and `mod_b` side by side in one `summary_col` table (use the cell above as a template). Does the table tell the same story you just told in words?

In [ ]:
# your code here

In the next session we are going to learn how to visually represent our data. See you then!